# Calculating trend rates

Calculates 12-month changes and fitted annual trend rates for each series, then compares cost trends with the premium trend

In [1]:
import sqlite3
import numpy as np
import pandas as pd

conn = sqlite3.connect("../data/trend.db")

In [2]:
#twelve month changes, matching on the date one year earlier, not 12 rows back, since insurance has no row for November 2025
#self join: two copies of the table (cur and prev) put each month next to the same month a year earlier
#left join keeps every current month, even if there's no match a year earlier; those months get a NULL instead of disappearing
#yoy is short for year over year: comparison with the same period one year earlier


yoy_query = """
SELECT cur.series,
       cur.date,
       cur.value,
       prev.value AS value_year_ago,
       cur.value / prev.value - 1 AS yoy_change
FROM monthly_series AS cur
LEFT JOIN monthly_series AS prev
  ON prev.series = cur.series
 AND prev.date = date(cur.date, '-1 year')
WHERE cur.date >= '2015-01-01'
ORDER BY cur.series, cur.date
"""
yoy = pd.read_sql(yoy_query, conn)
yoy.tail(12)

,series,date,value,value_year_ago,yoy_change
837,used_cars,2025-11-01,186.298,179.904,0.035541
838,used_cars,2025-12-01,184.668,181.705,0.016307
839,used_cars,2026-01-01,181.269,184.891,-0.019590
840,used_cars,2026-02-01,180.589,186.557,-0.031990
841,used_cars,2026-03-01,179.830,185.728,-0.031756
842,used_cars,2026-04-01,179.822,184.800,-0.026937
843,used_cars,2026-05-01,180.005,183.664,-0.019922
844,used_cars,2026-06-01,179.591,182.833,-0.017732
845,used_cars,2026-07-01,180.306,183.746,-0.018721
846,used_cars,2026-08-01,180.965,185.259,-0.023178


In [3]:
yoy[(yoy["series"] == "insurance") & (yoy["date"] >= "2024-01-01")].head()

,series,date,value,value_year_ago,yoy_change
108,insurance,2024-01-01,794.855,659.076,0.206014
109,insurance,2024-02-01,800.317,663.946,0.205395
110,insurance,2024-03-01,821.514,672.231,0.222071
111,insurance,2024-04-01,836.477,682.009,0.226490
112,insurance,2024-05-01,836.177,695.481,0.202300


In [4]:
yoy[(yoy["series"] == "insurance")].tail(3)

,series,date,value,value_year_ago,yoy_change
138,insurance,2026-08-01,849.837,895.769,-0.051277
139,insurance,2026-09-01,850.388,894.987,-0.049832
140,insurance,2026-10-01,856.439,NaN,NaN


In [5]:
latest = yoy.dropna(subset=["yoy_change"]).groupby("series").tail(1)
latest

,series,date,value,value_year_ago,yoy_change
139,insurance,2026-09-01,850.388,894.987,-0.049832
281,medical,2026-09-01,593.666,584.872,0.015036
422,miles_traveled,2026-08-01,278718.000,277668.000,0.003781
563,parts,2026-09-01,189.584,188.028,0.008275
705,repair,2026-09-01,468.491,442.843,0.057917
848,used_cars,2026-10-01,179.946,186.111,-0.033125


Premiums are down 5% from a year ago, while repair costs are up 5.8% and medical up 1.5%. Among the cost drivers, only used cars (-3.3%) are falling. If this continues, premiums will fall behind costs.

In [6]:
monthly_query = """
SELECT series, date, value
FROM monthly_series
WHERE date >= '2015-01-01'
  AND value IS NOT NULL
"""
monthly = pd.read_sql(monthly_query, conn)
monthly["date"] = pd.to_datetime(monthly["date"])
monthly.head()

,series,date,value
0,repair,2015-01-01,268.869
1,repair,2015-02-01,269.136
2,repair,2015-03-01,268.907
3,repair,2015-04-01,269.948
4,repair,2015-05-01,270.764


In [7]:
#check months per series (insurance is missing October and November 2025)
monthly["series"].value_counts()

series
used_cars         142
repair            141
parts             141
medical           141
miles_traveled    140
insurance         140
Name: count, dtype: int64

In [8]:
#fits an exponential trend over the latest window and returns the annual trend rate
#window will keeps only the last 3 or 5 years
def annual_trend(group, years):
    end = group["date"].max()
    window = group[group["date"] > end - pd.DateOffset(years=years)]
    t = (window["date"] - end).dt.days / 365.25
    slope = np.polyfit(t, np.log(window["value"]), 1)[0]
    return np.exp(slope) - 1

In [9]:
#run it for every series
rows = []
for name, group in monthly.groupby("series"):
    rows.append({"series": name,
                 "trend_3yr": annual_trend(group, 3),
                 "trend_5yr": annual_trend(group, 5)})

trends = pd.DataFrame(rows)
trends

,series,trend_3yr,trend_5yr
0,insurance,0.028132,0.105816
1,medical,0.026799,0.022663
2,miles_traveled,0.009991,0.011263
3,parts,0.021733,0.019424
4,repair,0.061314,0.074366
5,used_cars,-0.003687,-0.031108


Over 5 years, premiums outran every cost driver, showing that insurers caught up after costs spiked in 2021–22. Over 3 years, premiums slowed to +2.8%, while repair costs are still rising +6.1% a year. If that gap continues, premiums fall behind costs again. Used cars are negative over 5 years since the 5-year window starts near the 2021–22 used car price peak. Miles traveled is gaining around +1% a year, so exposure is growing steadily.

In [10]:
#insurance has 140 months, so check which months are blank (November 2025 has no row at all, found in notebook 1)

missing_query = """
SELECT date, value
FROM monthly_series
WHERE series = 'insurance'
  AND value IS NULL
"""
pd.read_sql(missing_query, conn)

,date,value
0,2025-10-01,None


Right now I have four separate cost trends. To answer our question, “Are premiums keeping up?”, I need one cost number to compare with the premium trend. To get that, I need a weighted average of the four severity trends, where the weights stand for roughly how much each one drives auto claim costs.

In [11]:
#severity weights based on CCC, Mitchell, LexisNexis, and Verisk figures (see README)
weights = {"repair": 0.31, "parts": 0.27, "used_cars": 0.17, "medical": 0.25}

trends["weight"] = trends["series"].map(weights)

severity_3yr = (trends["trend_3yr"] * trends["weight"]).sum()
severity_5yr = (trends["trend_5yr"] * trends["weight"]).sum()

Computing by hand: using weights of 31% repair, 27% parts, 17% used cars, and 25% medical, the combined severity trend is 3.09% a year over 3 years, against 2.81% for premiums, so premiums are roughly keeping pace but slipping slightly behind. Over 5 years, premiums grew 10.6% a year against 2.9% for costs, as insurers caught up after the 2021–22 cost spike.

In [12]:
#compare with premiums by Python, should match above
premium = trends.set_index("series").loc["insurance"]

comparison = pd.DataFrame({
    "window": ["3 years", "5 years"],
    "severity_trend": [severity_3yr, severity_5yr],
    "premium_trend": [premium["trend_3yr"], premium["trend_5yr"]]})
comparison["gap"] = comparison["premium_trend"] - comparison["severity_trend"]
comparison

,window,severity_trend,premium_trend,gap
0,3 years,0.030948,0.028132,-0.002816
1,5 years,0.028675,0.105816,0.077140


In [13]:
#sensitivity test
scenarios = {"Base": {"repair": 0.31, "parts": 0.27, "used_cars": 0.17, "medical": 0.25},
             "More injury": {"repair": 0.27, "parts": 0.23, "used_cars": 0.15, "medical": 0.35},
             "More total loss": {"repair": 0.26, "parts": 0.22, "used_cars": 0.27, "medical": 0.25},
             "More parts": {"repair": 0.25, "parts": 0.33, "used_cars": 0.17, "medical": 0.25}}

rows = []
for name, w in scenarios.items():
    sev_3yr = (trends["trend_3yr"] * trends["series"].map(w)).sum()
    rows.append({"scenario": name,
                 "severity_3yr": sev_3yr,
                 "gap_3yr": premium["trend_3yr"] - sev_3yr})

sensitivity = pd.DataFrame(rows)
sensitivity

,scenario,severity_3yr,gap_3yr
0,Base,0.030948,-0.002816
1,More injury,0.030380,-0.002248
2,More total loss,0.026427,0.001705
3,More parts,0.028573,-0.000441


Over 3 years, the gap between premium and severity trends stays within about 0.3 points of zero across all four weighting scenarios, and turns positive when total losses get more weight. So premiums have roughly kept pace with costs over 3 years. The stronger warning sign is the last 12 months, when premiums fell 5% while repair costs rose 5.8%, which the 3-year trend line hasn’t caught up to yet.

In [14]:
#trend factors

#trend factor = (1 + annual trend)^years, where years = middle of the data period to the middle of when new rates will be used
#trend period: midpoint of the latest 12 months (April 1, 2026) to the average claim date
#for 12-month policies written starting July 1, 2027 (July 1, 2028) = 2.25 years
trend_years = 2.25

comparison["severity_factor"] = (1 + comparison["severity_trend"]) ** trend_years
comparison["premium_factor"] = (1 + comparison["premium_trend"]) ** trend_years
comparison

,window,severity_trend,premium_trend,gap,severity_factor,premium_factor
0,3 years,0.030948,0.028132,-0.002816,1.070984,1.064413
1,5 years,0.028675,0.105816,0.077140,1.065679,1.253967


Projected over 2.25 years, costs would be about 7.1% higher using the 3-year severity trend, and premiums about 6.4% higher at their 3-year pace. The 3-year and 5-year severity factors are close, so the cost trend is stable. The 5-year premium factor (1.254) reflects insurers catching up after the 2021–22 cost spike, so rely on the 3-year premium trend for projecting forward.

In [15]:
#save results for charts and excel
trends.to_sql("trend_rates", conn, if_exists="replace", index=False)
comparison.to_sql("severity_vs_premium", conn, if_exists="replace", index=False)
sensitivity.to_sql("sensitivity", conn, if_exists="replace", index=False)
yoy.to_sql("yoy_changes", conn, if_exists="replace", index=False)
conn.close()

Over 5 years, premiums grew far faster than claim costs (10.6% vs. 2.9% a year) as insurers caught up after the 2021–22 cost spike; over 3 years, the two have roughly kept pace. The warning sign is the last 12 months, when premiums fell 5% while repair costs rose 5.8%, so if that continues, premiums will fall behind costs again.